In [9]:
import numpy as np

np.random.seed(42)

X=np.random.uniform(-2,2,(400,3))
y=np.sin(X[:,0])+0.5*(X[:,1]**2)-0.8*X[:,2]
y=y.reshape(-1,1)

X=X.T
y=y.T
N=X.shape[1]

In [10]:
def relu(z):
    return np.maximum(0,z)
def drelu(z):
    return (z>0).astype(float)

def sigmoid(z):
    return 1/(1+np.exp(-z))
def dsigmoid(z):
    s=sigmoid(z)
    return s*(1-s)

def tanh(z):
    return np.tanh(z)
def dtanh(z):
    return 1-np.tanh(z)**2

def leaky(z):
    return np.where(z>0,z,0.01*z)
def dleaky(z):
    return np.where(z>0,1,0.01)

def softplus(z):
    return np.log(1+np.exp(z))
def dsoftplus(z):
    return 1/(1+np.exp(-z))

In [11]:
acts={
"relu":(relu,drelu),
"sigmoid":(sigmoid,dsigmoid),
"tanh":(tanh,dtanh),
"leaky":(leaky,dleaky),
"softplus":(softplus,dsoftplus)
}

In [12]:
def init(layers):
    W=[]; b=[]
    for i in range(len(layers)-1):
        W.append(np.random.uniform(-0.5,0.5,(layers[i+1],layers[i])))
        b.append(np.zeros((layers[i+1],1)))
    return W,b

def forward(X,W,b,act):
    A=X
    Zs=[]; As=[A]
    for i in range(len(W)-1):
        Z=W[i]@A+b[i]
        A=act[0](Z)
        Zs.append(Z); As.append(A)
    Z=W[-1]@A+b[-1]
    A=Z
    Zs.append(Z); As.append(A)
    return A,Zs,As

def mse(y,yh):
    return np.mean((y-yh)**2)

def backward(y,yh,W,Zs,As,act):
    dW=[None]*len(W)
    db=[None]*len(W)
    dA=2*(yh-y)/N
    for i in reversed(range(len(W))):
        if i==len(W)-1:
            dZ=dA
        else:
            dZ=dA*act[1](Zs[i])
        dW[i]=dZ@As[i].T
        db[i]=np.sum(dZ,axis=1,keepdims=True)
        dA=W[i].T@dZ
    return dW,db

In [13]:
def train(layers,activation):

    W,b=init(layers)
    act=acts[activation]
    lr=0.01

    loss200=None

    for epoch in range(1000):

        yh,Zs,As=forward(X,W,b,act)
        loss=mse(y,yh)

        if epoch==200:
            loss200=loss

        dW,db=backward(y,yh,W,Zs,As,act)

        for i in range(len(W)):
            W[i]-=lr*dW[i]
            b[i]-=lr*db[i]

    grad_l1=np.linalg.norm(dW[0])
    grad_last=np.linalg.norm(dW[-2] if len(dW)>1 else dW[-1])

    print("Activation:",activation)
    print("Final Loss:",loss)
    print("Loss @200:",loss200)
    print("Grad Norm Layer1:",grad_l1)
    print("Grad Norm LastHidden:",grad_last)
    print()

In [14]:
models={
"A":[3,4,1],
"B":[3,6,6,1],
"C":[3,8,8,8,8,1],
"D":[3,8,8,8,8,8,8,8,8,1]
}

for m in models:
    print("Model",m)
    train(models[m],"relu")

print("Model D with Sigmoid")
train(models["D"],"sigmoid")

Model A
Activation: relu
Final Loss: 0.11154512827725767
Loss @200: 0.49268594052687364
Grad Norm Layer1: 0.04521736074874136
Grad Norm LastHidden: 0.04521736074874136

Model B
Activation: relu
Final Loss: 0.07288586596130892
Loss @200: 0.3205940688052462
Grad Norm Layer1: 0.036608745952825275
Grad Norm LastHidden: 0.021440804238454382

Model C
Activation: relu
Final Loss: 0.03045149820655308
Loss @200: 0.8477964004861142
Grad Norm Layer1: 0.023876345478441537
Grad Norm LastHidden: 0.01680066614352698

Model D
Activation: relu
Final Loss: 0.053184827185813396
Loss @200: 1.6326715841070876
Grad Norm Layer1: 0.42978396573665884
Grad Norm LastHidden: 0.6212899111786657

Model D with Sigmoid
Activation: sigmoid
Final Loss: 1.7438504812896145
Loss @200: 1.743850485090402
Grad Norm Layer1: 6.305806043555611e-06
Grad Norm LastHidden: 6.346814762002756e-06

